In [ ]:
# import requests

# API_URL = "https://router.huggingface.co/nscale/v1/chat/completions"
# headers = {
#     "Authorization": "Bearer ",
# }

# def query(payload):
#     response = requests.post(API_URL, headers=headers, json=payload)
#     return response.json()

# response = query({
#     "messages": [
#         {
#             "role": "user",
#             "content": [
#                 {
#                     "type": "text",
#                     "text": "Describe this image in one sentence."
#                 },
#                 {
#                     "type": "image_url",
#                     "image_url": {
#                         "url": "https://cdn.britannica.com/61/93061-050-99147DCE/Statue-of-Liberty-Island-New-York-Bay.jpg"
#                     }
#                 }
#             ]
#         }
#     ],
#     "model": "meta-llama/Llama-4-Scout-17B-16E-Instruct"
# })

# print(response["choices"][0]["message"])

In [ ]:
import requests
import base64

API_URL = "https://router.huggingface.co/nscale/v1/chat/completions"
headers = {
    "Authorization": "Bearer ",
}

# Step 1: Read local image and encode as base64
def encode_image_to_base64(image_path):
    with open(image_path, "rb") as image_file:
        encoded_string = base64.b64encode(image_file.read()).decode("utf-8")
    return encoded_string

# Step 2: Send request with base64 image
def query_with_local_image(image_path):
    image_base64 = encode_image_to_base64(image_path)
    
    payload = {
        "messages": [
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": """You are an expert academic transcript analyzer.

The uploaded image is a scanned student transcript. Your task is to extract and structure all relevant academic data into clean, natural-language format, categorized into clearly defined sections.

Please include the following categories in your response. Use bullet points or short paragraphs as appropriate. Do not replicate layout or table format — express everything in plain natural language.

---

### 1. Student Summary:
- Full Name of the Student
- College Name
- Advisors (if listed)
- Organization Name(if present)
- Degree or Major(s)
- Career Totals:
  - GPA
  - Hrs Att (Attempted Hours)
  - Hrs Ern (Earned Hours)
  - Hrs Gpa (GPA Hours)
  - Qual Pts (Quality Points)

---

### 2. Term-wise Academic Summary:
For each term (e.g., **Fall 2024**, **Spring 2025**, **Transfer Term**):

- **Term Name and Year**
- **Term Totals**:
  - GPA
  - Hrs Att
  - Hrs Ern
  - Hrs Gpa
  - Qual Pts

- **Subterm Summary** (if applicable):
  For each subterm (e.g., subterm1, subterm2), include:
  - Subterm Totals:
    - GPA
    - Hrs Att
    - Hrs Ern
    - Hrs Gpa
    - Qual Pts

---

### 3. Courses Summary:
List all courses under their respective **term** and **subterm** (if applicable). For each course, include the following details in natural language:

- Course Number
- Course Title
- CR Type (e.g., CR, TR, ND)
- Grade (e.g., A, B, W, WIP, NM)
- Rpt (if applicable, otherwise skip)
- Hrs Att (Attempted Hours)
- Hrs Ern (Earned Hours)
- Hrs Gpa (GPA Hours)
- Qual Pts (Quality Points)
- GPA (if listed)
- Completion Date (if available)

---

### 4. Transfer Courses (if any):
If transfer courses are listed (typically under a "Transfer Term"):
- Mention the source institution name
- For each course:
  - Course Number
  - Course Title
  - CR Type
  - Grade
  - Hrs Att
  - Hrs Ern
  - Hrs Gpa
  - Qual Pts

---

### Output Formatting Instructions:
- Write in clear and readable English, not code or JSON.
- Avoid unnecessary table layout or repetition.
- Skip empty or missing fields.
- Use grouped sections by term and subterm logically.
- Do not hallucinate information. Only include what’s visually available from the image.

"""
                    },
                    {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:image/png;base64,{image_base64}"
                        }
                    }
                ]
            }
        ],
        "model": "meta-llama/Llama-4-Scout-17B-16E-Instruct"
    }

    response = requests.post(API_URL, headers=headers, json=payload)
    return response.json()

# image_path = "data\\transcripts\\op_image\\barretttrista_1.png"
# response = query_with_local_image(image_path)

# # Print response
# print(response["choices"][0]["message"])


In [3]:
# response["choices"][0]["message"]['content']

In [4]:
import os
import json
import re

img_path_v1 = 'data\\transcripts\\op_image'
transcript_db = {}

# Function to extract name, college and store transcript
def store_transcript(content: str, storage_dict: dict) -> dict:
    """
    Extracts student name and college name from the transcript content
    and stores the full content in the dictionary using the key format:
    'StudentName_CollegeName'.
    """
    try:
        # Extract Full Name
        name_match = re.search(r"Full Name of the Student:\s*(.*)", content)
        college_match = re.search(r"College Name:\s*(.*)", content)

        if not name_match or not college_match:
            raise ValueError("Could not extract student name or college name from content.")

        student_name = name_match.group(1).strip().replace(" ", "_")
        college_name = college_match.group(1).strip().replace(" ", "_")

        key = f"{student_name}_{college_name}"
        storage_dict[key] = content.strip()

        return storage_dict

    except Exception as e:
        print(f"Error storing transcript: {e}")
        return storage_dict

# Loop through each image and process it
for i in os.listdir(img_path_v1):
    temp_path = os.path.join(img_path_v1, i)
    print(f"Processing: {temp_path}")
    
    try:
        response = query_with_local_image(temp_path)  # Your LLM image query function
        content = response["choices"][0]["message"]['content']
        
        # Store in transcript dictionary
        transcript_db = store_transcript(content, transcript_db)
    
    except Exception as e:
        print(f"Failed to process {temp_path}: {e}")

# Save the entire transcript data to JSON file
with open("data//transcripts_student.json", "w", encoding="utf-8") as f:
    json.dump(transcript_db, f, indent=2, ensure_ascii=False)

print("All transcripts saved successfully.")


Processing: data\transcripts\op_image\barretttrista_1.png
Processing: data\transcripts\op_image\barretttrista_2.png
Processing: data\transcripts\op_image\bezuworkbien_1.png
Processing: data\transcripts\op_image\brightlesline_1.png
Processing: data\transcripts\op_image\buchananchristian_1.png
Processing: data\transcripts\op_image\buchananchristian_2.png
Processing: data\transcripts\op_image\cavazosarnoldo_1.png
Processing: data\transcripts\op_image\gaitanjoshua_1.png
Processing: data\transcripts\op_image\gaitanjoshua_2.png
All transcripts saved successfully.


In [5]:
# import json
# import faiss
# import numpy as np
# from sentence_transformers import SentenceTransformer
# from typing import List, Dict, Tuple, Any, Union
# from langchain.text_splitter import RecursiveCharacterTextSplitter
# from langchain_groq import ChatGroq
# import re

# # ------------------- Helper Functions for Structured Data ------------------- #
# def extract_student_name_from_key(student_key: str) -> str:
#     """Extract student name from the structured key format"""
#     # Remove college name from the end
#     parts = student_key.split('_')
#     # Find where college name starts (usually contains "College" or common institution words)
#     college_indicators = ['College', 'University', 'Institute', 'Academy', 'School']
    
#     name_parts = []
#     for part in parts:
#         if any(indicator in part for indicator in college_indicators):
#             break
#         name_parts.append(part.replace('_', ' '))
    
#     return ' '.join(name_parts) if name_parts else student_key.replace('_', ' ')

# def extract_college_from_key(student_key: str) -> str:
#     """Extract college name from the structured key format"""
#     parts = student_key.split('_')
#     college_indicators = ['College', 'University', 'Institute', 'Academy', 'School']
    
#     college_parts = []
#     found_college = False
#     for part in parts:
#         if any(indicator in part for indicator in college_indicators):
#             found_college = True
#         if found_college:
#             college_parts.append(part.replace('_', ' '))
    
#     return ' '.join(college_parts) if college_parts else "Unknown College"

# def extract_meaningful_sections(student_key: str, content: str) -> List[Tuple[str, str]]:
#     """
#     Extract meaningful sections from structured student data
#     Returns: List of (text_chunk, metadata) tuples
#     """
#     chunks = []
#     student_name = extract_student_name_from_key(student_key)
#     college_name = extract_college_from_key(student_key)
    
#     # Split content by main sections
#     sections = re.split(r'### \d+\.', content)
    
#     for i, section in enumerate(sections):
#         if not section.strip():
#             continue
            
#         section = section.strip()
        
#         # Identify section type
#         if "Student Summary" in section or i == 1:
#             # Student Summary Section
#             text_chunk = f"Student: {student_name} at {college_name}. {section}"
#             chunks.append((text_chunk, f"{student_key}|student_summary"))
            
#         elif "Term-wise Academic Summary" in section:
#             # Term-wise Academic Summary
#             text_chunk = f"Student: {student_name}. Academic Summary: {section}"
#             chunks.append((text_chunk, f"{student_key}|academic_summary"))
            
#             # Split by individual terms
#             term_sections = re.split(r'#### ', section)
#             for term_section in term_sections[1:]:  # Skip first empty split
#                 if term_section.strip():
#                     term_name = term_section.split('\n')[0].strip('*').strip()
#                     term_text = f"Student: {student_name}. Term: {term_name}. {term_section}"
#                     chunks.append((term_text, f"{student_key}|term|{term_name}"))
                    
#         elif "Courses Summary" in section:
#             # Courses Summary Section
#             text_chunk = f"Student: {student_name}. Courses Information: {section}"
#             chunks.append((text_chunk, f"{student_key}|courses_summary"))
            
#             # Split by individual terms in courses
#             course_term_sections = re.split(r'#### ', section)
#             for course_term_section in course_term_sections[1:]:  # Skip first empty split
#                 if course_term_section.strip():
#                     term_name = course_term_section.split('\n')[0].strip('*').strip()
                    
#                     # Extract individual courses
#                     course_matches = re.findall(r'- \*\*([^*]+)\*\*:(.*?)(?=\n- \*\*|\n####|\Z)', course_term_section, re.DOTALL)
                    
#                     for course_code, course_details in course_matches:
#                         course_text = f"Student: {student_name}. Term: {term_name}. Course: {course_code.strip()}. Details: {course_details.strip()}"
#                         chunks.append((course_text, f"{student_key}|course|{term_name}|{course_code.strip()}"))
                    
#                     # Also create a term-level course summary
#                     term_courses_text = f"Student: {student_name}. Term: {term_name}. All Courses: {course_term_section}"
#                     chunks.append((term_courses_text, f"{student_key}|term_courses|{term_name}"))
                    
#         elif "Transfer Courses" in section:
#             # Transfer Courses Section
#             text_chunk = f"Student: {student_name}. Transfer Information: {section}"
#             chunks.append((text_chunk, f"{student_key}|transfer_courses"))
    
#     return chunks

# # ------------------- Step 1: Direct Text Extraction from Structured Data ------------------- #
# def extract_text_from_structured_data(all_student_data: Dict[str, str]) -> List[Tuple[str, str]]:
#     """
#     Extract meaningful text chunks directly from structured student data
#     Returns: List of (text_chunk, metadata) tuples
#     """
#     all_chunks = []
    
#     for student_key, content in all_student_data.items():
#         # Extract meaningful sections
#         student_chunks = extract_meaningful_sections(student_key, content)
#         all_chunks.extend(student_chunks)
    
#     return all_chunks

# # ------------------- Step 2: Improved Chunking (Same as Original) ------------------- #
# def create_text_chunks(all_student_data: Dict[str, str], chunk_size=500, chunk_overlap=100) -> Tuple[List[str], List[str]]:
#     """
#     Create text chunks with better context preservation
#     """
#     all_texts = []
#     all_metadata = []
    
#     # Extract meaningful chunks first
#     meaningful_chunks = extract_text_from_structured_data(all_student_data)
    
#     for text, metadata in meaningful_chunks:
#         # Split large chunks if needed
#         if len(text) > chunk_size:
#             splitter = RecursiveCharacterTextSplitter(
#                 chunk_size=chunk_size, 
#                 chunk_overlap=chunk_overlap,
#                 separators=[". ", "\n", "- ", ", ", " ", ""]
#             )
#             sub_chunks = splitter.split_text(text)
#             for i, sub_chunk in enumerate(sub_chunks):
#                 all_texts.append(sub_chunk)
#                 all_metadata.append(f"{metadata}|chunk_{i}")
#         else:
#             all_texts.append(text)
#             all_metadata.append(metadata)
    
#     return all_texts, all_metadata

# # ------------------- Step 3: Embeddings + Indexing (Same as Original) ------------------- #
# def create_embeddings(texts: List[str], model_name="all-MiniLM-L6-v2"):
#     """Create embeddings for text chunks"""
#     model = SentenceTransformer(model_name)
#     embeddings = model.encode(texts, show_progress_bar=True)
#     return embeddings, model

# def create_faiss_index(embeddings: np.ndarray):
#     """Create FAISS index for similarity search"""
#     dim = embeddings.shape[1]
#     index = faiss.IndexFlatL2(dim)
#     index.add(embeddings)
#     return index

# # ------------------- Step 4: Enhanced Search (Same as Original) ------------------- #
# def search(query: str, model, index, all_texts: List[str], all_metadata: List[str], top_k=10):
#     """Enhanced search with better result filtering"""
#     query_vec = model.encode([query])
#     distances, indices = index.search(np.array(query_vec), top_k)
    
#     # Group results by student
#     student_matches = {}
    
#     print("\n🔍 Top Matches from FAISS Index:")
#     for i, idx in enumerate(indices[0]):
#         if idx < len(all_texts):
#             metadata = all_metadata[idx]
#             student_key = metadata.split('|')[0]
#             text_preview = all_texts[idx][:200] + "..." if len(all_texts[idx]) > 200 else all_texts[idx]
            
#             print(f"  {i+1}. Student: {extract_student_name_from_key(student_key)}")
#             print(f"     Text: {text_preview}")
#             print(f"     Distance: {distances[0][i]:.4f}")
#             print()
            
#             if student_key not in student_matches:
#                 student_matches[student_key] = []
#             student_matches[student_key].append({
#                 'text': all_texts[idx],
#                 'metadata': metadata,
#                 'distance': distances[0][i]
#             })
    
#     return student_matches

# # ------------------- Step 5: Comprehensive Summary Creation ------------------- #
# def create_comprehensive_summary(student_key: str, content: str) -> str:
#     """Create a comprehensive summary from structured content"""
#     student_name = extract_student_name_from_key(student_key)
#     college_name = extract_college_from_key(student_key)
    
#     lines = []
#     lines.append(f"=== STUDENT INFORMATION ===")
#     lines.append(f"Name: {student_name}")
#     lines.append(f"College: {college_name}")
#     lines.append("")
    
#     # Add the structured content directly
#     lines.append(f"=== DETAILED ACADEMIC INFORMATION ===")
#     lines.append(content)
    
#     return "\n".join(lines)

# # ------------------- Step 6: LLM Response Generation (Enhanced) ------------------- #
# def generate_answer(user_query: str, student_matches: Dict, full_data: Dict[str, str]):
#     """Generate comprehensive answer using LLM"""
    
#     # Prepare context from matched students
#     context_parts = []
#     for student_key, matches in student_matches.items():
#         if student_key in full_data:
#             summary = create_comprehensive_summary(student_key, full_data[student_key])
#             context_parts.append(summary)
    
#     context = "\n\n" + "="*100 + "\n\n".join(context_parts)
    
#     prompt = f"""You are an expert academic advisor assistant. Based on the student transcript data provided below, answer the user's question accurately and comprehensively.

# IMPORTANT INSTRUCTIONS:
# 1. Use ONLY the information provided in the academic data below
# 2. Be specific about course codes, titles, grades, credit hours, and GPA information
# 3. If asking about a specific student, identify them by their full name clearly
# 4. Organize your response with clear formatting and structure
# 5. Include relevant details like term information, course performance, and academic progress
# 6. If information is not available or unclear, state that explicitly
# 7. When discussing courses, include both course codes and titles when available
# 8. Mention GPA, credit hours, and academic standing when relevant

# ACADEMIC DATA:
# {context}

# USER QUESTION: {user_query}

# ANSWER:"""

#     llm = ChatGroq(
#         model="llama3-8b-8192",
#         temperature=0.1,
#         max_tokens=4000,
#         timeout=60,
#         max_retries=2,
#     )
    
#     response = llm.invoke(prompt)
#     return response.content

# # ------------------- Step 7: Main Search Function (Enhanced) ------------------- #
# def search_and_generate(user_query: str, model, index, all_texts: List[str], all_metadata: List[str], full_data: Dict[str, str], top_k=15):
#     """Main function to search and generate response"""
    
#     print(f"🔎 Searching for: '{user_query}'")
#     print("=" * 80)
    
#     # Search for relevant information
#     student_matches = search(user_query, model, index, all_texts, all_metadata, top_k=top_k)
    
#     if not student_matches:
#         print("❌ No matching students found.")
#         return
    
#     print(f"📊 Found information for {len(student_matches)} student(s)")
#     print("=" * 80)
    
#     # Generate comprehensive answer
#     answer = generate_answer(user_query, student_matches, full_data)
    
#     print("🧠 FINAL ANSWER:")
#     print("=" * 80)
#     print(answer)
#     print("=" * 80)

# # ------------------- Step 8: Main Execution Function ------------------- #
# def main():
#     """Main function to run the RAG system"""
    
#     with open("data//transcripts_student.json", "r") as f:
#         full_data = json.load(f) 
    
#     # Load your actual data here
#     print("📚 Loading structured student data...")
    
#     # Create text chunks with improved context
#     print("\n📝 Creating text chunks from structured data...")
#     all_texts, all_metadata = create_text_chunks(full_data)
#     print(f"✅ Created {len(all_texts)} text chunks")
    
#     # Create embeddings
#     print("\n🔄 Creating embeddings...")
#     embeddings, model = create_embeddings(all_texts)
#     print(f"✅ Created embeddings with shape: {embeddings.shape}")
    
#     # Create FAISS index
#     print("\n🗂️ Creating FAISS index...")
#     index = create_faiss_index(np.array(embeddings))
#     print("✅ FAISS index created successfully!")
    
#     # Test queries
#     test_queries = [
#         # "Tell me about Trista Denay Barrett's courses and academic performance",
#         # "What courses is Bien Tadesse Bezuwork taking?",
#         # "What is Bien Tadesse Bezuwork's GPA and academic standing?",
#         # "Show me the courses for Spring 2025",
#         # "Which students are studying Biology?",
#         "Tell me the courses which Arnoldo Bernal Cavazos has enrolled?",
#         "Tell me the courses which Joshua Don Gaitan has enrolled?",
#         "Tell me the courses which Leslie Nichole Bright has enrolled?",
#     ]
    
#     print("\n" + "="*100)
#     print("🎯 TESTING QUERIES")
#     print("="*100)
    
#     for query in test_queries:
#         print(f"\n🔍 QUERY: {query}")
#         print("-" * 80)
#         search_and_generate(query, model, index, all_texts, all_metadata, full_data)
#         print("\n" + "="*100)

# if __name__ == "__main__":
#     main()

In [15]:
import json
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from typing import List, Dict, Tuple, Any, Union
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

# ------------------- Step 1: Direct Text Chunking ------------------- #
def create_text_chunks(all_student_data: Dict[str, str], chunk_size=500, chunk_overlap=100) -> Tuple[List[str], List[str]]:
    """
    Create text chunks directly from student data
    """
    all_texts = []
    all_metadata = []
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap,
        separators=[". ", "\n", "- ", ", ", " ", ""]
    )
    
    for student_key, content in all_student_data.items():
        # Split the content into chunks
        chunks = splitter.split_text(content)
        
        for i, chunk in enumerate(chunks):
            # Add student context to each chunk
            contextualized_chunk = f"Student ID: {student_key}\n{chunk}"
            all_texts.append(contextualized_chunk)
            all_metadata.append(f"{student_key}|chunk_{i}")
    
    return all_texts, all_metadata

# ------------------- Step 2: Embeddings + Indexing ------------------- #
def create_embeddings(texts: List[str], model_name="all-MiniLM-L6-v2"):
    """Create embeddings for text chunks"""
    model = SentenceTransformer(model_name)
    embeddings = model.encode(texts, show_progress_bar=True)
    return embeddings, model

def create_faiss_index(embeddings: np.ndarray):
    """Create FAISS index for similarity search"""
    dim = embeddings.shape[1]
    index = faiss.IndexFlatL2(dim)
    index.add(embeddings)
    return index

# ------------------- Step 3: Search Function ------------------- #
def search(query: str, model, index, all_texts: List[str], all_metadata: List[str], top_k=10):
    """Search for relevant chunks using FAISS"""
    query_vec = model.encode([query])
    distances, indices = index.search(np.array(query_vec), top_k)
    
    results = []
    
    # print("\n🔍 Top Matches from FAISS Index:")
    for i, idx in enumerate(indices[0]):
        if idx < len(all_texts):
            metadata = all_metadata[idx]
            student_key = metadata.split('|')[0]
            text_preview = all_texts[idx][:2000] + "..." if len(all_texts[idx]) > 2000 else all_texts[idx]
            
            print(f"  {i+1}. Student: {student_key}")
            print(f"     Text: {text_preview}")
            print(f"     Distance: {distances[0][i]:.4f}")
            print()
            
            results.append({
                'text': all_texts[idx],
                'metadata': metadata,
                'distance': distances[0][i],
                'student_key': student_key
            })
    
    return results

def extract_final_answer(response_text: str) -> str:
    """Extract content after 'FINAL ANSWER:' keyword"""
    
    # List of possible final answer keywords to check
    final_answer_keywords = [
        "FINAL ANSWER:",
        "Final Answer:",
        "final answer:",
        "ANSWER:",
        "Answer:",
        "answer:"
    ]
    
    # Try to find any of the keywords
    for keyword in final_answer_keywords:
        if keyword in response_text:
            # Split by the keyword and take everything after it
            parts = response_text.split(keyword, 1)
            if len(parts) > 1:
                # Clean up the extracted answer
                final_answer = parts[1].strip()
                
                # Remove any trailing separators or extra formatting
                final_answer = final_answer.replace("=" * 80, "").strip()
                
                return final_answer
    
    # If no keyword found, return the original response
    return response_text

# ------------------- Step 4: LLM Response Generation ------------------- #
def generate_answer(user_query: str, search_results: List[Dict]):
    """Generate comprehensive answer using LLM"""
    
    # Prepare context from search results
    context_parts = []
    for result in search_results:
        context_parts.append(result['text'])
    
    context = "\n\n".join(context_parts)
    
    with open("data\prompt.txt", "r", encoding="utf-8") as file:
        loaded_prompt = file.read()

    prompt = loaded_prompt.format(context=context, user_query=user_query)

    llm = ChatGroq(
        model="llama3-8b-8192",
        temperature=0.1,
        max_tokens=4000,
        timeout=60,
        max_retries=2,
    )
    
    response = llm.invoke(prompt)
    final_answer = extract_final_answer(response.content)
    return final_answer

# ------------------- Step 5: Main Search Function ------------------- #
def search_and_generate(user_query: str, model, index, all_texts: List[str], all_metadata: List[str], top_k=15):
    """Main function to search and generate response"""
    
    print(f"🔎 Searching for: '{user_query}'")
    print("=" * 80)
    
    # Search for relevant information
    search_results = search(user_query, model, index, all_texts, all_metadata, top_k=top_k)
    
    if not search_results:
        print("❌ No matching results found.")
        return
    
    print(f"📊 Found {len(search_results)} relevant chunks")
    print("=" * 80)
    
    # Generate comprehensive answer
    answer = generate_answer(user_query, search_results)
    
    print("🧠 FINAL ANSWER:")
    print("=" * 80)
    print(answer)
    print("=" * 80)

# ------------------- Step 6: Main Execution Function ------------------- #
def main():
    """Main function to run the RAG system"""
    
    # Load data from JSON file
    print("📚 Loading student data...")
    with open("data\\transcripts_student.json", "r") as f:
        full_data = json.load(f)
    
    print(f"✅ Loaded data for {len(full_data)} students")
    
    # Create text chunks
    print("\n📝 Creating text chunks...")
    all_texts, all_metadata = create_text_chunks(full_data)
    print(f"✅ Created {len(all_texts)} text chunks")
    
    # Create embeddings
    print("\n🔄 Creating embeddings...")
    embeddings, model = create_embeddings(all_texts)
    print(f"✅ Created embeddings with shape: {embeddings.shape}")
    
    # Create FAISS index
    print("\n🗂️ Creating FAISS index...")
    index = create_faiss_index(np.array(embeddings))
    print("✅ FAISS index created successfully!")
    
    # Test queries
    test_queries = [
        # "Tell me the courses which Arnoldo Bernal Cavazos has enrolled?",
        # "Tell me the courses which Joshua Don Gaitan has enrolled?",
        "Tell me the courses which Leslie Nichole Bright has enrolled?",
        # "How many Students have A grade in Fall 2024-2025 and their details",
        # "List of students from Murray State College",
        # "List of students from NEWMAN UNIVERSITY",
        # "Sort students in descending order of GPA"
    ]
    
    print("\n" + "="*100)
    print("🎯 TESTING QUERIES")
    print("="*100)
    
    for query in test_queries:
        print(f"\n🔍 QUERY: {query}")
        print("-" * 80)
        search_and_generate(query, model, index, all_texts, all_metadata)
        print("\n" + "="*100)

if __name__ == "__main__":
    main()

📚 Loading student data...
✅ Loaded data for 7 students

📝 Creating text chunks...
✅ Created 38 text chunks

🔄 Creating embeddings...


Batches: 100%|██████████| 2/2 [00:00<00:00,  3.92it/s]


✅ Created embeddings with shape: (38, 384)

🗂️ Creating FAISS index...
✅ FAISS index created successfully!

🎯 TESTING QUERIES

🔍 QUERY: Tell me the courses which Leslie Nichole Bright has enrolled?
--------------------------------------------------------------------------------
🔎 Searching for: 'Tell me the courses which Leslie Nichole Bright has enrolled?'
  1. Student: Leslie_Nichole_Bright_Hesston_College
     Text: Student ID: Leslie_Nichole_Bright_Hesston_College
### 1. Student Summary:
- Full Name of the Student: Leslie Nichole Bright
- College Name: Hesston College
- Advisors: Laura Lyndsey
- Degree or Major(s): Bachelor of Arts, Business Management BA Degree
- Career Totals:
  - GPA: 0.00
  - Hrs Att (Attempted Hours): 14.00
  - Hrs Ern (Earned Hours): 0.00
  - Hrs Gpa (GPA Hours): 14.00
  - Qual Pts (Quality Points): 0.00

### 2
     Distance: 0.7650

  2. Student: Trista_Denay_Barrett_Hesston_College
     Text: Student ID: Trista_Denay_Barrett_Hesston_College
. Courses Summar

### Switch between policy types and student transcript types

In [ ]:
from sentence_transformers import SentenceTransformer, util

model = SentenceTransformer('all-MiniLM-L6-v2')

transcript_examples = [
    "What is the student's GPA?",
    "Tell me the academic information of the student",
    "Which term and sub term is the course taken?",
    "Give me the types of courses the student is pursuing",
    "Show me the career total",
    "What is the student's credit hour total?",
    "How has the student performed in each semester?",
]

# Precompute transcript embeddings
transcript_embeddings = model.encode(transcript_examples, convert_to_tensor=True)

# Your user query
user_question = "Can you tell me the types of courses student is pursuing?"

# Embed the user question
user_embedding = model.encode(user_question, convert_to_tensor=True)

# Compute cosine similarities with transcript references
cos_scores = util.cos_sim(user_embedding, transcript_embeddings)

# Get the highest similarity score
max_score = cos_scores.max().item()

# Threshold for classification
SIMILARITY_THRESHOLD = 0.6

# Final classification logic
if max_score >= SIMILARITY_THRESHOLD:
    print("Classified as: STUDENT TRANSCRIPT TYPE")
else:
    print("Classified as: POLICY TYPE")
